# AI Powered HR Assistant - RAG

In [ ]:
!pip install langchain langchain-openai openai langchain-community PyPDF faiss-cpu chromadb tiktoken langchain-chroma langchain_experimental

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.8/85.8 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 330.6/330.6 kB 37.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.4/21.4 MB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.1/210.1 kB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 33.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 92.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 49.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 500.1/500.1 kB 52.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 44.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
# from langchain_chroma import Chroma

from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

In [ ]:
import gradio as gr

In [ ]:
pdf_path = '1728286846_the_nestle_hr_policy_pdf_2012.pdf'
loader = PyPDFLoader(pdf_path)
docs = loader.load()

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=200,
)

chunk = splitter.split_documents(docs)

In [ ]:
from google.colab import userdata
api_key = userdata.get('OPENAI_API_KEY')

embedding = OpenAIEmbeddings(model = "text-embedding-3-large", openai_api_key = api_key)
vectorstore_FAISS = FAISS.from_documents(chunk, embedding)

In [ ]:
retriever = vectorstore_FAISS.as_retriever(search_kwargs={"k": 5})

In [ ]:
def format_docs(docs):
    return "\n\n".join(
        f"[page = {d.metadata.get("page","?")}] {d.page_content}"
        for d in docs
        )

In [ ]:
rag_prompt = ChatPromptTemplate.from_messages([
    ("system",
     """You are an AI-powered HR assistant designed to help Nestlé employees quickly and accurately understand company HR policies by answering questions clearly, professionally, and based strictly on official HR policy documents.
      When answering HR policy questions:
      - Use clear section headings or bullet points when appropriate
      - Keep the tone professional and employee-friendly
      - Avoid repeating long policy language verbatim
      - Structure answers for readability and quick understanding
     Using ONLY the provided HR policy context, answer the question below.
      Format your response as:
      - A short summary sentence
      - Followed by bullet points grouped by theme
      - Use clear, business-friendly language
      - If you are nearing the output limit, finish the current sentence and end with a short summary.
     """
     "Also while generating you follow these RULES:"
     "1. Do NOT use Outside Knowledge"
     "2. Try to limit your responses to 200 words or less."),
    ("human",
     "Context:\n{context}\n\nQuestion:\n{question}\n\nAnswer:")
])

In [ ]:
model_RAG = ChatOpenAI(
    model="gpt-4",
    temperature=0,
    max_tokens=500,
    openai_api_key=api_key)

parser = StrOutputParser()

In [ ]:
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | rag_prompt
    | model_RAG
    | parser
)

In [ ]:
# For sanity checking retreiver results
query = "What is Nestle's policy on career development for employees?"
results = retriever.invoke(query)

for i, doc in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
The Nestlé Human Resources Policy
4
Learning is part of the Company culture.
Employees at all levels are systematically 
encouraged to consider how they upgrade their 
knowledge and skills.
The Company determines training and deve-
lopment priorities. The responsibility for turning 
these into actions is shared between employees, 
line managers and the Human Resources. 
Experience and on-the-job training are the 
primary source of learning. Managers are 
responsible for guiding and coaching employees 
to succeed in their current positions.  
Nestlé employees understand the importance 
of continuous improvement, as well as sharing 
knowledge and ideas freely with others. Practices 
such as lateral professional development, 
extension of responsibilities, and cross functional

--- Result 2 ---
company. As such, Nestlé has focused on remo-
ving barriers to career progression for women 
and men by developing a more flexible work 
environment, initiating mentoring schemes,

In [ ]:
# Test 1 - Raw outputs - no Gradio

# Out of scope question for the RAG

query = "What is machine learning"

answer = rag_chain.invoke(query)

print(answer)


Your question about machine learning is not related to the provided HR policy context. The Nestlé Human Resources Policy does not provide information on machine learning. Please provide a question related to the HR policy for an accurate response.


In [ ]:
# Test 2 - Raw outputs - no Gradio

# In scope question for the RAG

query = "What is Nestle's policy on career development for employees?"

answer = rag_chain.invoke(query)

print(answer)


Nestlé's policy on career development emphasizes continuous learning, shared responsibility, and opportunities for international assignments.

Continuous Learning and Development:
- Learning is a part of Nestlé's company culture, with employees at all levels encouraged to upgrade their knowledge and skills.
- The company determines training and development priorities, with the responsibility for action shared between employees, line managers, and Human Resources.
- Experience and on-the-job training are the primary sources of learning, with managers guiding and coaching employees.

Career Progression and Opportunities:
- Promotions are based on sustained performance and future potential, with an active succession planning process in place.
- The company offers lateral professional development, extension of responsibilities, and cross-functional assignments.
- Employees are encouraged to express their career objectives and expectations in an open dialogue with their line managers.

Inte

In [ ]:
# With Gradio
import gradio as gr

def query_hr_document(query):
    return rag_chain.invoke(query)  # make sure this returns a markdown string

with gr.Blocks(title="Nestlé HR Policy Assistant") as demo:
    gr.Markdown("# 🧑‍💼 Nestlé HR Policy Assistant")

    query_input = gr.Textbox(
        label="HR Question",
        placeholder="e.g. What is Nestlé's policy on career development?",
        lines=3
    )

    answer_output = gr.Markdown()

    submit_btn = gr.Button("Ask")

    submit_btn.click(query_hr_document, inputs=query_input, outputs=answer_output)

demo.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://bf4902938bcda578a3.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
